# dfs-three-set-toposort — faded example 3: Three-set DFS skips already-processed shared nodes via the perm set

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `dfs-three-set-toposort`. Running the beacon reports progress on the `Backprop: DFS three-set toposort` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """A minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries an optional `.recipe`
    populated by wrap_forward_fn. `requires_grad` is set by the wrapper.
    `.grad` accumulates the leaf gradient at the end of the reverse pass."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: DFS three-set toposort` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`dfs-three-set-toposort`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "dfs-three-set-toposort"
DD_SUBTOPIC = "Backprop: DFS three-set toposort"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

In any DAG with shared descendants (convergent paths), the DFS toposort must emit each node exactly once. This is guaranteed by the `perm` set: once a node has been fully processed and appended to `result`, its `id` lives in `perm`, and any subsequent call to `visit` for that node returns immediately without re-processing or re-appending.

## Faded exercise 3

Complete `topological_sort_shared`. Three source nodes all depend on the same two shared leaf nodes. Fill in only the `perm`-check early-return at the top of `visit` — the rest of the skeleton is provided.

**Fill in:** The early-return guard that checks if the node's id is already in `perm` and returns without doing anything if so.

In [ ]:
def topological_sort_shared(root, get_children):
    result = []
    perm = set()
    temp = set()

    def visit(node):
        nid = id(node)
        if nid in perm:
            return
        if nid in temp:
            raise ValueError('Cycle detected')
        temp.add(nid)
        for child in get_children(node):
            visit(child)
        temp.discard(nid)
        perm.add(nid)
        result.append(node)

    visit(root)
    return result


def _test():
    class N:
        def __init__(self, name, kids=None): self.name = name; self.kids = kids or []
        def __repr__(self): return self.name

    # shared1 and shared2 are referenced by three parent nodes
    shared1 = N('S1')
    shared2 = N('S2')
    p1 = N('P1', [shared1, shared2])
    p2 = N('P2', [shared1, shared2])
    p3 = N('P3', [shared1, shared2])
    root = N('R', [p1, p2, p3])

    result = topological_sort_shared(root, lambda n: n.kids)

    # 6 distinct nodes, each exactly once
    assert len(result) == 6
    assert len(set(id(n) for n in result)) == 6

    # shared nodes appear before parent nodes
    for shared in [shared1, shared2]:
        for parent in [p1, p2, p3]:
            assert result.index(shared) < result.index(parent)

    # root is last
    assert result[-1] is root


try:
    _test()
    _dd_passed.add('faded3')
    print('[Delta Drills] faded3 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
def topological_sort_shared(root, get_children):
    result = []
    perm = set()
    temp = set()

    def visit(node):
        nid = id(node)
        if nid in perm:
            return
        if nid in temp:
            raise ValueError('Cycle detected')
        temp.add(nid)
        for child in get_children(node):
            visit(child)
        temp.discard(nid)
        perm.add(nid)
        result.append(node)

    visit(root)
    return result
```
</details>